In [1]:
import pandas as pd
import os
from pathlib import Path

import sys
sys.path.append('../src') # add src directory to path

from utils import format_discussion

## Dataset Exploration

In [2]:
# parent directory
parent_dir = Path.cwd().parent

# path to the dataset
data_path = os.path.join(parent_dir ,'data','raw','ParlEE_UK_plenary_speeches.csv')

# reading a CSV file into a DataFrame
df = pd.read_csv(data_path)

C:\Users\FedeF\AppData\Local\Temp\ipykernel_19400\2812760694.py:8: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_path)


In [3]:
df.head()

,instance_id,date,agenda,speechnumber,sentencenumber,speaker,party,text,parliament,iso3country,chair,eu,policyarea,cmp_party
0,1,12/01/2009,Defence,1,1,Simon Hughes,LibDem,What recent assessment he has made of the risk...,UK-HouseOfCommons,GBR,False,0,16,51421.0
1,2,12/01/2009,Defence,2,1,John Hutton,Lab,"Before I begin, I am sure that the whole House...",UK-HouseOfCommons,GBR,False,0,19,51320.0
2,3,12/01/2009,Defence,2,2,John Hutton,Lab,"They were Sergeant Christopher Reed, of 6th Ba...",UK-HouseOfCommons,GBR,False,0,16,51320.0
3,4,12/01/2009,Defence,2,3,John Hutton,Lab,Our thoughts and prayers are likewise with the...,UK-HouseOfCommons,GBR,False,0,19,51320.0
4,5,12/01/2009,Defence,2,4,John Hutton,Lab,Providing effective help and support for servi...,UK-HouseOfCommons,GBR,False,0,16,51320.0


### Formatting discussions


To identify a single debate in our dataset, we decided to consider the features 'agenda' and 'date'. In our dataset, each entry represents a single sentence said in a debate by a politician. Through the function ```format_discussion()``` we're able to join all sentences to reconstruct the debate.

In [4]:
sample_discussion = format_discussion(df, agenda="Defence", date="12/01/2009")
print(sample_discussion)

Simon Hughes (LibDem): What recent assessment he has made of the risk of overstretch affecting forces' families.
John Hutton (Lab): Before I begin, I am sure that the whole House will wish to join me in sending our profound condolences to the families and friends of the servicemen killed in Afghanistan since the House last met. They were Sergeant Christopher Reed, of 6th Battalion the Rifles, and Corporal Robert Deering, Corporal Liam Elms and Lance Corporal Ben Whatley, all of them Royal Marines. Our thoughts and prayers are likewise with the family of the Royal Marine who died in Afghanistan yesterday. Providing effective help and support for service families remains a high priority for the Ministry of Defence. We recognise the challenges that service families face as a result of the current high level of operations and have already made a significant investment-for example, in service accommodation and in making it easier for service personnel and their families to keep in touch dur

Now i need to create a new DataFrame, where for each agenda-date pair, it has a 'discussion' entry that contains the whole discussion. I need to use the 'format_discussion()' method.

Since ParlEE_eng is more than 1.5 GB, this task has to be performed in a very efficient way.

In [ ]:

# create a smaller dataframe with relevant columns
df_temp = df[['date', 'agenda', 'speechnumber', 'sentencenumber', 'speaker', 'party', 'text']]
df_temp.head()

,date,agenda,speechnumber,sentencenumber,speaker,party,text
0,12/01/2009,Defence,1,1,Simon Hughes,LibDem,What recent assessment he has made of the risk...
1,12/01/2009,Defence,2,1,John Hutton,Lab,"Before I begin, I am sure that the whole House..."
2,12/01/2009,Defence,2,2,John Hutton,Lab,"They were Sergeant Christopher Reed, of 6th Ba..."
3,12/01/2009,Defence,2,3,John Hutton,Lab,Our thoughts and prayers are likewise with the...
4,12/01/2009,Defence,2,4,John Hutton,Lab,Providing effective help and support for servi...


In [22]:
# create new dataset with formatted discussions
formatted_discussions = []

# only one loop to group by agenda and date
for (agenda, date), group in df_temp.groupby(['agenda', 'date']):
    discussion_text = format_discussion(group, agenda, date)
    formatted_discussions.append({
        'agenda': agenda,
        'date': date,
        'discussion': discussion_text
    })

discussion_df = pd.DataFrame(formatted_discussions)


In [23]:
# save the df to a csv file
output_path = os.path.join(parent_dir ,'data','processed','discussion_dataset.csv')
discussion_df.to_csv(output_path, index=False)

In [ ]:
print(discussion_df[(discussion_df['agenda'] == 'Defence') & (discussion_df['date'] == "12/01/2009")]['discussion'].values[0])

# compare to the original sample discussion


Simon Hughes (LibDem): What recent assessment he has made of the risk of overstretch affecting forces' families.
John Hutton (Lab): Before I begin, I am sure that the whole House will wish to join me in sending our profound condolences to the families and friends of the servicemen killed in Afghanistan since the House last met. They were Sergeant Christopher Reed, of 6th Battalion the Rifles, and Corporal Robert Deering, Corporal Liam Elms and Lance Corporal Ben Whatley, all of them Royal Marines. Our thoughts and prayers are likewise with the family of the Royal Marine who died in Afghanistan yesterday. Providing effective help and support for service families remains a high priority for the Ministry of Defence. We recognise the challenges that service families face as a result of the current high level of operations and have already made a significant investment-for example, in service accommodation and in making it easier for service personnel and their families to keep in touch dur

## Classification

The point is: we don't actually need to use all debates, this would slow down the entire process. To lighten the dataset, we will only consider Debates that include **opinion-based sentences**.

Then we will summarize the single sentences to make them more comparable. The summarization in fact, takes away noise from the speech. This noise can be introduced through tone, word choice, etc.

In [4]:
from DatasetFilterer import DatasetFilterer
import json


# path to the records file
record_path = os.path.join(parent_dir ,'data','external','records.json')

with open(record_path, 'r', encoding="utf-8") as f:
    records = json.load(f)


uk = DatasetFilterer(speech=df, records=records)
topic = "Gaza"

record = uk.get_records()

# TODO: optionally, if we wanna further speed up the process, we can try using polars module
filtered_df = uk.filter_speeches(topic)

c:\Users\FedeF\Documents\GitHub\stance-detection-eu-parliaments\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\FedeF\Documents\GitHub\stance-detection-eu-parliaments\venv\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.7.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [5]:
filtered_df.head()

,instance_id,date,agenda,speechnumber,sentencenumber,speaker,party,text,parliament,iso3country,chair,eu,policyarea,cmp_party
53,54,12/01/2009,Iran [Oral Answers to Questions > Defence > Wr...,15,3,Denis MacShane,Lab,"It stated:â€œHamas, with training from Iran an...",UK-HouseOfCommons,GBR,False,0,19,51320.0
55,56,12/01/2009,Iran [Oral Answers to Questions > Defence > Wr...,15,5,Denis MacShane,Lab,Does the Secretary of State agree that Iran's ...,UK-HouseOfCommons,GBR,False,0,19,51320.0
409,410,12/01/2009,Gaza,106,8,David Miliband,Lab,Hamas refused to extend the lull and instead f...,UK-HouseOfCommons,GBR,False,0,16,51320.0
410,411,12/01/2009,Gaza,106,9,David Miliband,Lab,"Those rockets, and the hundreds fired since, w...",UK-HouseOfCommons,GBR,False,0,16,51320.0
445,446,12/01/2009,Gaza,106,44,David Miliband,Lab,Those armaments are the source of fear for hun...,UK-HouseOfCommons,GBR,False,0,16,51320.0


In [6]:
print(df.shape)
print(filtered_df.shape)

(6767026, 14)
(4147, 14)


In [7]:
# save the filtered df to a csv file
output_filtered_path = os.path.join(parent_dir ,'data','processed',f'filtered_speeches_{topic}.csv')
filtered_df.to_csv(output_filtered_path, index=False)

In [8]:
classified_df = uk.classify_sentences(topic)

In [9]:
classified_df.head()

,instance_id,date,agenda,speechnumber,sentencenumber,speaker,party,text,parliament,iso3country,chair,eu,policyarea,cmp_party,classification
409,410,12/01/2009,Gaza,106,8,David Miliband,Lab,Hamas refused to extend the lull and instead f...,UK-HouseOfCommons,GBR,False,0,16,51320.0,opinion
410,411,12/01/2009,Gaza,106,9,David Miliband,Lab,"Those rockets, and the hundreds fired since, w...",UK-HouseOfCommons,GBR,False,0,16,51320.0,opinion
446,447,12/01/2009,Gaza,106,45,David Miliband,Lab,They are also a threat to any prospect of Pale...,UK-HouseOfCommons,GBR,False,0,19,51320.0,opinion
495,496,12/01/2009,Gaza,107,10,William Hague,Con,I am sure that the Foreign Secretary will agre...,UK-HouseOfCommons,GBR,False,0,19,51620.0,opinion
496,497,12/01/2009,Gaza,107,11,William Hague,Con,Does the Foreign Secretary agree that it is ne...,UK-HouseOfCommons,GBR,False,0,16,51620.0,opinion


In [10]:
# save the classified df to a csv file
output_classified_path = os.path.join(parent_dir ,'data','processed',f'classified_speeches_{topic}.csv')
classified_df.to_csv(output_classified_path, index=False)

## Summarization

Now that we have the opinion-based sentences, we can proceed to summarize them, in order to remove noise introduced by word choice, sentiment, etc...

Here is an example of how can we use an LLM to summarize a whole debate

In [10]:
import ollama 

client = ollama.Client()

model = "llama3.2"
prompt = f"Summarize the following UK parliamentary debate discussion in a concise manner:\n\n{sample_discussion}\n\nSummary:"

response = client.chat(model=model, messages=[{"role": "user", "content": prompt}])

print("Ollama Response:")
print(response.message['content'])

Ollama Response:
The UK parliamentary debate discussion focused on the challenges faced by service families due to the high level of operations and overstretch affecting forces. The Secretary of State for Defence (John Hutton) expressed condolences to the families of servicemen killed in Afghanistan and reiterated the government's commitment to providing effective help and support.

Liberal Democrat MP Simon Hughes asked about recent assessments of the risk of overstretch affecting forces' families, given the upcoming change in Iraq and the potential reduction in operational tempo. The Secretary of State acknowledged the opportunity to provide better support and promised to continue looking at measures to help families.

Other MPs raised concerns about housing provision for service personnel when they leave the armed forces, the need for greater clarity regarding accommodation options, and the importance of providing morale-boosting information to families while loved ones are on activ

TODO: use ROUGE score to evaluate summary

After summarizing the sentences, we can proceed to map them on an embedding space!